In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd 


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from typing import TypedDict, Annotated, Optional, List, Literal
from pydantic import BaseModel, EmailStr, Field

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_core.runnables import ConfigurableField
from langchain_core.tools import tool

In [ ]:
Amodel = ChatAnthropic(
    model='claude-sonnet-4-5-20250929',
    anthropic_api_key="")

In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

#### Local Tool 

##### Tool Defining

In [ ]:
@tool
def multiply(x: int, y: int) -> int:
    """Multiply 'x' times 'y'."""
    return x * y
    

In [ ]:
multiply.invoke({"x": 5,"y": 6})

In [ ]:
llm = HuggingFacePipeline.from_model_id('microsoft/Phi-3-mini-4k-instruct', task='text-generation')
model = ChatHuggingFace(llm = llm)

In [ ]:
@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the 'y'."""
    return x**y
@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

In [ ]:
print(add.name)
print(add.description)
print(add.args)

##### Tool Binding

In [ ]:
tools = [multiply, exponentiate, add]

In [ ]:
tool_model = Amodel.bind_tools(tools)

In [ ]:
query = "What is 393 * 12.25? Also, what is 11 + 49? also divide 10 by 5"
messages = [HumanMessage(query)]
ai_msg = tool_model.invoke(messages)
messages.append(ai_msg)
ai_msg

In [ ]:
tools[0]

In [ ]:
tools[0].args_schema.model_json_schema()

In [ ]:
multiply.invoke(ai_msg.tool_calls[0])

In [ ]:
ai_msg.tool_calls

In [ ]:
messages = []
query = "What is 393 * 12? Also, what is 11 + 49?"
messages = [HumanMessage(query)]
ai_msg = tool_model.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply,
    "exponentiate": exponentiate}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    print(f"{tool_msg.name} {tool_call['args']} {tool_msg.content}")
    print(tool_msg)
    messages.append(tool_msg)
final_response = tool_model.invoke(messages)
print(final_response.content)

In [ ]:
final_response.content

#### API-Based Tool

###### Example : 1

In [ ]:

@tool
def get_weather(latitude: float, longitude: float) -> str:
    """
    Get current weather using Open-Meteo API.
    Provide latitude and longitude of the location.
    """
    
    url = "https://api.open-meteo.com/v1/forecast"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        return "Error fetching weather data."

    data = response.json()
    weather = data.get("current_weather", {})

    return (
        f"Temperature: {weather.get('temperature')}°C\n"
        f"Wind Speed: {weather.get('windspeed')} km/h\n"
        f"Weather Code: {weather.get('weathercode')}\n"
        f"Time: {weather.get('time')}"
    )

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
# Initialize the LLM with GPT-4o and bind the tools
tools = [multiply, exponentiate, add, wiki]
tool_model = Amodel.bind_tools(tools)
messages = [HumanMessage("What was the most impressive thing about Imran Khan?. and add 5 with 10 and then divide the answer with 8")]
ai_msg = tool_model.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    tool = {"wikipedia":wiki, "add": add, "multiply": multiply,
    "exponentiate": exponentiate}[tool_call["name"].lower()]
    tool_msg = tool.invoke(tool_call)
    print(tool_msg.name)
    print(tool_call['args'])
    print(tool_msg.content)
    messages.append(tool_msg)
    print()
final_response = tool_model.invoke(messages)
print(final_response.content)

In [ ]:
messages

###### Example : 2

In [ ]:
@tool
def get_stock_price(ticker: str) -> float:
"""Get the stock price for the stock exchange ticker for the company."""
api_url = f"https://api.example.com/stocks/{ticker}"
response = requests.get(api_url)
if response.status_code == 200:
    data = response.json()
    return data["price"]
else:
    raise ValueError(f"Failed to fetch stock price for {ticker}")

    
# Initialize the LLM with GPT-4o and bind the tools
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools([get_stock_price])

messages = [HumanMessage("What is the stock price of Apple?")]

ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_msg = get_stock_price.invoke(tool_call)
    print(tool_msg.name)
    print(tool_call['args'])
    print(tool_msg.content)
    messages.append(tool_msg)
    print()
    
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

##### Plugin Tools

##### MCP

##### Statefull Tools